# 07 – Multi-Agent Training (Risk + Fairness + Portfolio Agents)

**Project:** Multi-Agent DRL + CNN Alternative Data for Credit Decisioning

Three specialized agents:
- **Risk Agent**: Minimizes expected default cost
- **Fairness Agent**: Improves outcomes for thin-file applicants
- **Portfolio Agent**: Balances long-term portfolio health

A coordinator combines their outputs (RQ1).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
X = np.load(DATA_PROCESSED / "X_fused.npy")
y = np.load(DATA_PROCESSED / "y.npy")
thin = np.load(DATA_PROCESSED / "thin.npy")

STATE_DIM = X.shape[1]
N_ACTIONS = 2
print(f"State dim: {STATE_DIM}")

In [ ]:
class CreditEnv:
    def __init__(self, X, y, thin):
        self.X = X
        self.y = y
        self.thin = thin
        self.n = len(y)
        self.idx = 0

    def reset(self):
        self.idx = np.random.randint(0, self.n)
        return self.X[self.idx]

    def step(self, action):
        true_label = self.y[self.idx]
        is_thin = self.thin[self.idx]

        if action == 0:  # Approve
            if true_label == 0:
                reward = 1.0 + (0.8 if is_thin else 0.0)
            else:
                reward = -5.0
        else:  # Reject
            if true_label == 1:
                reward = 5.0
            else:
                reward = -1.0 - (0.5 if is_thin else 0.0)

        done = True
        self.idx = (self.idx + 1) % self.n
        return self.X[self.idx], reward, done, {"thin": is_thin, "true_label": true_label}

env = CreditEnv(X, y, thin)

In [ ]:
class SpecializedAgent(nn.Module):
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 2)
        )
    def forward(self, x):
        return self.net(x)

class MultiAgentCoordinator(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.risk_agent = SpecializedAgent(state_dim)
        self.fairness_agent = SpecializedAgent(state_dim)
        self.portfolio_agent = SpecializedAgent(state_dim)
        self.combine = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        r = self.risk_agent(x)
        f = self.fairness_agent(x)
        p = self.portfolio_agent(x)
        combined = torch.cat([r, f, p], dim=-1)
        return self.combine(combined)

marl = MultiAgentCoordinator(STATE_DIM).to(device)
optimizer = optim.Adam(marl.parameters(), lr=3e-4)
print(f"Total parameters: {sum(p.numel() for p in marl.parameters()):,}")

In [ ]:
EPISODES = 1000
rewards_history = []
thin_approval_history = []

for ep in range(1, EPISODES + 1):
    state = env.reset()
    total_r = 0
    thin_approvals = 0
    thin_count = 0

    for t in range(40):
        state_t = torch.tensor(state, dtype=torch.float32, device=device)
        logits = marl(state_t)
        dist = Categorical(logits=logits)
        action = dist.sample()

        next_state, reward, done, info = env.step(action.item())
        
        loss = -dist.log_prob(action) * reward
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_r += reward
        if info["thin"]:
            thin_count += 1
            if action.item() == 0:
                thin_approvals += 1

        state = next_state
        if done:
            break

    rewards_history.append(total_r)
    thin_rate = thin_approvals / max(thin_count, 1)
    thin_approval_history.append(thin_rate)

    if ep % 150 == 0:
        avg_r = np.mean(rewards_history[-100:])
        avg_thin = np.mean(thin_approval_history[-100:])
        print(f"Episode {ep:4d} | Avg Reward: {avg_r:6.2f} | Thin-file Approval: {avg_thin:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(pd.Series(rewards_history).rolling(50).mean())
axes[0].set_title("Multi-Agent – Smoothed Reward")
axes[0].set_xlabel("Episode")

axes[1].plot(pd.Series(thin_approval_history).rolling(50).mean(), color="green")
axes[1].set_title("Thin-file Approval Rate")
axes[1].set_xlabel("Episode")

plt.tight_layout()
plt.savefig(RESULTS / "marl_training.png", dpi=120)
plt.show()

torch.save(marl.state_dict(), RESULTS / "multi_agent_coordinator.pt")
print("Saved → results/multi_agent_coordinator.pt")